In [3]:
import pandas as pd
import numpy as np
from ta import add_all_ta_features
from tigramite import data_processing as pp
from tigramite.independence_tests import ParCorr
from tigramite.pcmci import PCMCI
import matplotlib.pyplot as plt
import seaborn as sns



# 1️⃣ Read CSV
path = "C:/Users/yhsol/Downloads/AMZN_alpha_1min_2024-05_to_2025-05 (1).csv"
df = pd.read_csv(path)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df.set_index('Timestamp', inplace=True)

# Add all TA features
df_indicators = add_all_ta_features(
    df.copy(),
    open="Open", high="High", low="Low", close="Close", volume="Volume",
    fillna=True
)

# Shift all features to avoid leakage
df_features = df_indicators.shift(1)

# Define the prediction target
df_features["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)  # for binary classification

# Drop rows with NaNs introduced by shifting
df_features = df_features.dropna()


c:\Users\yhsol\AppData\Local\Programs\Python\Python312\Lib\site-packages\ta\trend.py:1030: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  self._psar[i] = high2


In [4]:
# Drop original columns if you want only indicators + Close price for causality
target_variable = 'Close'
columns_to_keep = [target_variable] + [col for col in df_indicators.columns if col != target_variable]
df_indicators = df_indicators[columns_to_keep]


In [ ]:
# Drop any remaining NaNs (if any)
df_indicators.dropna(inplace=True)

# 3️⃣ Prepare Data for Tigramite
dataframe = pp.DataFrame(df_indicators.values)
# 4️⃣ Define PCMCI object
parcorr = ParCorr(significance='analytic')
pcmci = PCMCI(dataframe=dataframe, cond_ind_test=parcorr)
# 5️⃣ Run PCMCI+
results = pcmci.run_pcmci(tau_max=5, pc_alpha=0.05)

In [ ]:
# 6️⃣ Examine p-values
q_matrix = pcmci.get_corrected_pvalues(p_matrix=results['p_matrix'], fdr_method='fdr_bh')
close_index = columns_to_keep.index(target_variable)
val_matrix = results['val_matrix']
causal_links = []
for i, col in enumerate(columns_to_keep):
    for lag in range(1, 6):
        p_val = q_matrix[i, close_index, lag-1]
        if p_val <= 0.05:
            causal_links.append((col, f"Lag {lag}", p_val, val_matrix[i, close_index, lag-1]))

# Sort by strength descending
causal_links_sorted = sorted(causal_links, key=lambda x: abs(x[3]), reverse=True)

print("Top Causal Indicators affecting 'Close':")
for link in causal_links_sorted:
    print(f"{link[0]} at {link[1]} | p-value: {link[2]:.4f}, strength: {link[3]:.4f}")


Top Causal Indicators affecting 'Close':
trend_ema_fast at Lag 1 | p-value: 0.0000, strength: 1.0000
trend_ema_slow at Lag 1 | p-value: 0.0000, strength: 1.0000
others_cr at Lag 1 | p-value: 0.0000, strength: 1.0000
trend_macd_signal at Lag 1 | p-value: 0.0000, strength: 0.9432
trend_macd at Lag 1 | p-value: 0.0000, strength: 0.9427
trend_trix at Lag 1 | p-value: 0.0000, strength: 0.9224
momentum_ppo_hist at Lag 1 | p-value: 0.0000, strength: 0.9211
momentum_ppo at Lag 1 | p-value: 0.0000, strength: 0.9205
momentum_ppo_signal at Lag 1 | p-value: 0.0000, strength: 0.9205
others_dr at Lag 1 | p-value: 0.0000, strength: 0.9186
others_dlr at Lag 1 | p-value: 0.0000, strength: 0.9173
trend_sma_fast at Lag 1 | p-value: 0.0000, strength: 0.8997
momentum_tsi at Lag 1 | p-value: 0.0000, strength: 0.8969
volatility_kcp at Lag 1 | p-value: 0.0000, strength: 0.8962
momentum_rsi at Lag 1 | p-value: 0.0000, strength: 0.8882
momentum_roc at Lag 1 | p-value: 0.0000, strength: 0.8878
High at Lag 1 | p-

In [6]:
import pandas as pd
import os
from pathlib import Path

# Convert to DataFrame
df_causal_links = pd.DataFrame(causal_links_sorted, columns=["Feature", "Lag", "p-value", "Strength"])

# Get Downloads path
downloads_path = str(Path.home() / "Downloads")
file_path = os.path.join(downloads_path, "causal_links_close.csv")

# Save to Downloads
df_causal_links.to_csv(file_path, index=False)

print(f"Causal links saved to: {file_path}")


Causal links saved to: C:\Users\yhsol\Downloads\causal_links_close.csv
